In [1]:

import pyspark
import pandas as pd
import warnings
warnings.filterwarnings(action='ignore')

In [2]:
import findspark
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.feature import StringIndexer
from pyspark.ml import Pipeline
from pyspark.ml.clustering import KMeans
from pyspark.ml.evaluation import ClusteringEvaluator
from pyspark.ml.evaluation import MulticlassClassificationEvaluator
from pyspark.sql import SparkSession
from pyspark.sql.types import DoubleType

In [3]:
findspark.init('/Applications/spark-3.3.1-bin-hadoop3')

In [4]:
from pyspark.sql import SparkSession
spark=SparkSession.builder.appName('SparkLab').getOrCreate()

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


23/01/05 17:48:28 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [5]:
data = spark.read.load("./data/Restaurant/df_final_dataset2.csv", 
                       format="csv", sep=",", header="true", inferSchema=True)

In [6]:
data.show()

+-----------+------+----------+-----------+--------+---------------+---------+----+
|customer_id|gender|item_count|grand_total|is_rated|vendor_rating_x|vendor_id|rank|
+-----------+------+----------+-----------+--------+---------------+---------+----+
|          0|  male|         1|        5.2|     Yes|              5|      582|  11|
|          0|  male|         1|        5.2|     Yes|              5|      582|  11|
|          0|  male|         1|        3.5|     Yes|              5|      582|  11|
|          0|  male|         2|        6.3|     Yes|              5|      582|  11|
|          0|  male|         4|       15.0|     Yes|              5|      582|  11|
|          0|  male|         5|       16.0|     Yes|              5|      582|  11|
|          0|  male|         2|        5.7|     Yes|              5|      582|  11|
|          0|  male|         1|        5.2|     Yes|              5|      582|  11|
|          0|  male|         1|        5.2|     Yes|              5|      58

In [7]:
[trainingData, testData] = data.randomSplit([0.7, 0.3])

In [8]:
assembler = VectorAssembler(
    inputCols = ["customer_id", "item_count", "grand_total", "vendor_rating_x", "vendor_id", "rank"],
    outputCol = "features"
)

In [9]:
indexer = StringIndexer(inputCol="rank", outputCol="label")

In [10]:
kmeans = KMeans().setK(2).setSeed(1)

In [11]:
pipeline = Pipeline(stages=[assembler, indexer, kmeans])

In [12]:
model = pipeline.fit(trainingData)

In [13]:
prediction = model.transform(testData)

prediction.show()

+-----------+------+----------+-----------+--------+---------------+---------+----+--------------------+-----+----------+
|customer_id|gender|item_count|grand_total|is_rated|vendor_rating_x|vendor_id|rank|            features|label|prediction|
+-----------+------+----------+-----------+--------+---------------+---------+----+--------------------+-----+----------+
|          0|  male|         1|        3.5|     Yes|              5|      582|  11|[0.0,1.0,3.5,5.0,...|  0.0|         0|
|          0|  male|         1|        3.5|     Yes|              5|      582|  11|[0.0,1.0,3.5,5.0,...|  0.0|         0|
|          0|  male|         1|        3.5|     Yes|              5|      582|  11|[0.0,1.0,3.5,5.0,...|  0.0|         0|
|          0|  male|         1|        5.2|     Yes|              5|      582|  11|[0.0,1.0,5.2,5.0,...|  0.0|         0|
|          0|  male|         1|        5.2|     Yes|              5|      582|  11|[0.0,1.0,5.2,5.0,...|  0.0|         0|
|          0|  male|    

### Evaluate KMeans Classification results

In [14]:
evaluator = MulticlassClassificationEvaluator(
    labelCol = 'label', predictionCol = 'prediction', metricName='accuracy'
)

p = prediction.withColumn("prediction", prediction["prediction"].cast("double"))

accuracy = evaluator.evaluate(p)

print("Classification Error = %g" % (1.0 - accuracy))

Classification Error = 0.395622


### Show cluster centres

In [15]:
centres = model.stages[-1].clusterCenters()

print("Cluster Centres:")
for centre in centres:
    print(centre)

Cluster Centres:
[764.88622499   2.43356515  16.35065886   4.55014641 267.25829673
  10.14531479]
[3.16692540e+03 2.16955458e+00 1.40180127e+01 4.38678414e+00
 2.55319922e+02 9.68330886e+00]


In [16]:
spark.stop()